# Solar PV Forecasting Using Satellite Data & ML

Forecasts PV solar output using real satellite-derived irradiance data
(NREL Himawari NSRDB), a physics-based PV model, and Prophet time-series
forecasting. See `README.md` for setup and the standalone dashboard
(`app.py`).

> **Note:** NREL (National Renewable Energy Laboratory) was renamed the "National Laboratory of the Rockies" (NLR) by the US Department of Energy on December 1, 2025. The developer portal moved from developer.nrel.gov to **developer.nlr.gov** — the API paths themselves are unchanged, only the domain. If you signed up for a key before the rename, it should still work at the new domain.

## 1. Fetch data from the NREL API

**Never hardcode your API key.** Set it as an environment variable
before starting Jupyter:

```bash
export NREL_API_KEY="your_key_here"   # macOS/Linux
setx NREL_API_KEY "your_key_here"     # Windows (new terminal needed after)
```

Get a free key at https://developer.nlr.gov/signup/

In [ ]:
import os
import requests

API_KEY = os.environ.get("NREL_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "Set the NREL_API_KEY environment variable before running this "
        "cell -- see the markdown cell above for how."
    )

EMAIL = os.environ.get("NREL_EMAIL")
if not EMAIL:
    raise RuntimeError(
        "Set the NREL_EMAIL environment variable to your real email "
        "before running this cell -- the API rejects placeholder "
        "addresses. e.g. os.environ['NREL_EMAIL'] = 'you@example.com'"
    )

url = "https://developer.nlr.gov/api/nsrdb/v2/solar/himawari-download.csv"
params = {
    "api_key": API_KEY,
    "email": EMAIL,
    "wkt": "POINT(75.9235 22.5297)",  # IIT Indore campus (longitude latitude)
    "names": "2019",
    "leap_day": "false",
    "interval": "10",
    "utc": "false",
    "mailing_list": "false",
}

response = requests.get(url, params=params)

# Previously this wrote response.content straight to disk regardless of
# whether the request actually succeeded -- an error response (e.g. bad
# email, invalid key, rate limit) would silently become a ".csv" file
# that's actually JSON, with no indication anything went wrong until
# you tried to load it later. Check the status and content first.
if response.status_code != 200 or response.headers.get("Content-Type", "").startswith("application/json"):
    print("Request failed -- response was not a CSV file:")
    print(response.text[:1000])
    raise RuntimeError(f"NLR API request failed with status {response.status_code}")

with open("himawari_2019_data.csv", "wb") as f:
    f.write(response.content)

print("Download complete: himawari_2019_data.csv")


## 2. Install dependencies

In [ ]:
!pip install requests prophet pandas matplotlib scikit-learn streamlit

## 3. Load data and reconstruct the timestamp

The raw NREL Himawari export has a 2-line metadata header, then
separate `Year`/`Month`/`Day`/`Hour`/`Minute` columns instead of a
single timestamp column.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("himawari_2019_data.csv", skiprows=2)
print(df.columns.tolist())

df['Timestamp'] = pd.to_datetime(dict(
    year=df['Year'], month=df['Month'], day=df['Day'],
    hour=df['Hour'], minute=df['Minute']
))
df.set_index('Timestamp', inplace=True)

plt.figure(figsize=(12, 5))
df['GHI'].plot(label='GHI')
plt.title("Global Horizontal Irradiance (GHI)")
plt.ylabel("W/m²")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df['GHI'].resample('D').mean().plot(figsize=(12, 4), title='Daily Average GHI')
plt.ylabel('W/m²')
plt.xlabel('Date')
plt.show()

## 4. Physics-based PV output model

In [ ]:
# System specifications
panel_area = 1.6          # m^2 per panel
panel_efficiency = 0.18   # 18%
num_panels = 10
derating_factor = 0.85    # wiring, inverter, and other real-world losses

df['PV_output_kw'] = (
    df['GHI'] * panel_area * num_panels * panel_efficiency * derating_factor
) / 1000  # W -> kW

daily_energy = df['PV_output_kw'].resample('D').sum()
daily_energy.plot(figsize=(12, 4), title="Estimated Daily PV Output (kWh)")
plt.ylabel("Energy (kWh)")
plt.xlabel("Date")
plt.show()

print(f"Estimated annual output: {daily_energy.sum():,.0f} kWh")

## 5. Temperature-derated model

Panel efficiency drops as cell temperature rises above 25°C (STC) --
this refines the flat-efficiency model above.

In [ ]:
df['T_cell'] = df['Temperature'] + (df['GHI'] / 800) * 20  # simple NOCT-style estimate
beta = 0.004  # ~0.4% efficiency loss per °C above 25

df['eff_temp'] = panel_efficiency * (1 - beta * (df['T_cell'] - 25))
df['PV_output_kw_temp'] = (
    df['GHI'] * panel_area * num_panels * df['eff_temp'] * derating_factor
) / 1000

daily_energy_temp = df['PV_output_kw_temp'].resample('D').sum()
print(f"Base model annual total:        {daily_energy.sum():,.0f} kWh")
print(f"Temperature-adjusted annual total: {daily_energy_temp.sum():,.0f} kWh")

## 6. Forecast with Prophet

Fit on daily-resampled output (fitting Prophet directly on 10-minute
data is unnecessarily slow and the day/night cycle dominates any
sub-daily seasonality Prophet would try to learn).

In [ ]:
from prophet import Prophet

df_prophet = daily_energy.reset_index()
df_prophet.columns = ['ds', 'y']

# Yearly seasonality needs ~2 years of history to fit reliably --
# Prophet will warn (not error) if given less, as here with 1 year.
model = Prophet(yearly_seasonality=True)
model.fit(df_prophet)

future = model.make_future_dataframe(periods=14, freq='D')
forecast = model.predict(future)

model.plot(forecast)
plt.title("Solar PV Output Forecast (Prophet)")
plt.ylabel("PV Output (kWh/day)")
plt.show()

## 7. Interactive dashboard

The Streamlit dashboard lives in a separate file, `app.py` -- Streamlit
apps must be launched from a terminal, they can't run as notebook cells:

```bash
streamlit run app.py
```

## Limitations

- The temperature-derating model uses a simplified NOCT-style cell
  temperature estimate, not a full thermal model
- Prophet's yearly seasonality is unreliable with only 1 year of
  history (it will warn about this) -- forecasts improve with 2+ years
  of historical data
- An LSTM model was explored but isn't included here in working form
  -- noted as a possible future direction, not a current feature